# AI Photo Enhancement
**Reflection Removal + NAFNet Denoise/Deblur + Real-ESRGAN Upscale**

This notebook runs the full application with a web UI accessible via a public URL.

**Requirements:**
- GPU runtime (T4 free tier works, A100/L4 recommended)
- HuggingFace token with access to `Qwen/Qwen-Image-Edit-2509`

---

## 1. Check GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected")

SyntaxError: unexpected character after line continuation character (1674539126.py, line 5)

## 2. Clone Repository & Install Dependencies

In [ ]:
import os
os.chdir('/content')

# Clone the repo
if not os.path.exists('AI-photo-enhancement'):
    !git clone https://github.com/buianhhuy96/AI-photo-enhancement.git
else:
    !cd AI-photo-enhancement && git pull

os.chdir('/content/AI-photo-enhancement')
print("\n✓ Repository ready")

In [ ]:
# Install Python dependencies
!pip install -q fastapi uvicorn[standard] python-multipart pydantic \
    torch torchvision diffusers transformers accelerate \
    safetensors huggingface_hub peft bitsandbytes \
    imageio numpy Pillow tqdm basicsr realesrgan gfpgan

print("\n✓ Python packages installed")

## 3. Build Frontend

In [ ]:
# Install Node.js and build the React frontend
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1
!node --version

os.chdir('/content/AI-photo-enhancement/web')
!npm install --legacy-peer-deps 2>&1 | tail -3
!npx vite build 2>&1 | tail -5
os.chdir('/content/AI-photo-enhancement')

print("\n✓ Frontend built to web/dist/")

## 4. Set HuggingFace Token & Download Models

You need a HuggingFace token with access to `Qwen/Qwen-Image-Edit-2509`.
Get one at https://huggingface.co/settings/tokens

In [ ]:
from google.colab import userdata
import os

# Option 1: Use Colab Secrets (recommended)
# Add a secret named 'HF_TOKEN' in the key icon on the left sidebar
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("✓ Using HF_TOKEN from Colab Secrets")
except Exception:
    # Option 2: Paste your token here
    os.environ['HF_TOKEN'] = ''  # <-- paste your token here if not using Secrets
    if not os.environ['HF_TOKEN']:
        print("⚠ No token set! Add HF_TOKEN to Colab Secrets or paste it above.")
    else:
        print("✓ Using pasted HF_TOKEN")

# Login to HuggingFace
!huggingface-cli login --token $HF_TOKEN 2>&1 | tail -2

In [ ]:
# Download model weights (~10GB total)
os.chdir('/content/AI-photo-enhancement')
!python3 download_models.py

## 5. Start Server with Public URL

This starts the FastAPI server and provides a URL via Colab's built-in proxy.
Click the URL that appears to open the web UI in your browser.


In [ ]:
import subprocess
import time
import os
import urllib.request
import fcntl

os.chdir("/content/AI-photo-enhancement")

# Kill any existing processes
!fuser -k 8000/tcp 2>/dev/null; sleep 2

# Set CUDA memory config for better fragmentation handling
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Start the FastAPI server in background
server_proc = subprocess.Popen(
    ["python3", "-u", "serve.py", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True
)

# Make stdout non-blocking so we can read while checking health
flags = fcntl.fcntl(server_proc.stdout, fcntl.F_GETFL)
fcntl.fcntl(server_proc.stdout, fcntl.F_SETFL, flags | os.O_NONBLOCK)

# Wait for server, printing its output as it loads
print("Waiting for server to start (model loading may take 1-3 min)...")
server_ready = False
for attempt in range(90):
    time.sleep(2)
    try:
        while True:
            line = server_proc.stdout.readline()
            if line:
                print(f"  [server] {line.strip()}")
            else:
                break
    except (IOError, BlockingIOError):
        pass
    if server_proc.poll() is not None:
        print(f"ERROR: Server exited with code {server_proc.returncode}")
        break
    try:
        urllib.request.urlopen("http://localhost:8000/api/settings/status", timeout=2)
        print(f"\nServer ready! (took {(attempt+1)*2}s)")
        server_ready = True
        break
    except Exception:
        pass

if not server_ready:
    print("Server failed to start. See output above.")
    raise SystemExit("Server startup failed")

# Use Colab built-in proxy (reliable, no DNS issues)
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8000)")
print(f"\n" + "="*60)
print(f"  App is live!")
print(f"  URL: {url}")
print("="*60)

# Keep cell alive - print server logs
print("\nServer running. Interrupt to stop.")
fcntl.fcntl(server_proc.stdout, fcntl.F_SETFL, flags)
try:
    while True:
        line = server_proc.stdout.readline()
        if not line:
            if server_proc.poll() is not None:
                print("Server process exited!")
                break
            time.sleep(0.1)
            continue
        print(f"[server] {line.strip()}")
except KeyboardInterrupt:
    print("\nStopping...")
    server_proc.terminate()
    print("Done.")


---
## Alternative: Mock Mode (no GPU needed, for UI testing only)

If you just want to test the UI without AI processing:

In [ ]:
# # Uncomment to run in mock mode (no AI, just UI testing)
# import subprocess, time, re
# os.chdir('/content/AI-photo-enhancement')
# server_proc = subprocess.Popen(
#     ['python3', 'serve.py', '--mock', '--port', '8000'],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
# )
# time.sleep(3)
# tunnel_proc = subprocess.Popen(
#     ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
# )
# for _ in range(30):
#     line = tunnel_proc.stdout.readline()
#     if line:
#         match = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
#         if match:
#             print(f"\n✓ Mock UI: {match.group(1)}\n")
#             break